In [3]:
!git clone https://github.com/JVTRILOKESH28/amazon-ml-2026-entity-resolution.git

Cloning into 'amazon-ml-2026-entity-resolution'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 32 (delta 8), reused 27 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 35.56 KiB | 367.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.


In [4]:
%cd amazon-ml-2026-entity-resolution

/content/amazon-ml-2026-entity-resolution


In [5]:
!find . -maxdepth 3 -type f

./.gitignore
./tests/test_blocking.py
./tests/test_preprocessing.py
./tests/test_features.py
./tests/test_evaluation.py
./docs/Documentation_template.md
./.git/description
./.git/HEAD
./.git/info/exclude
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-receive.sample
./.git/hooks/post-update.sample
./.git/hooks/applypatch-msg.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/pre-commit.sample
./.git/hooks/update.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/sendemail-validate.sample
./.git/hooks/commit-msg.sample
./.git/hooks/prepare-commit-msg.sample
./.git/packed-refs
./.git/logs/HEAD
./.git/index
./.git/config
./requirements.txt
./pyproject.toml
./models/.gitkeep
./notebooks/02_validation.ipynb
./notebooks/03_error_analysis.ipynb
./notebooks/01_eda.ipynb
./README.md
./src/business_entity_resolution.egg-info/SOURCES.txt
./src/business_entity_resolution.egg-info/depe

In [6]:
!pip install -e .

Obtaining file:///content/amazon-ml-2026-entity-resolution
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for business-entity-resolution (pyproject.toml) ... done
  Created wheel for business-entity-resolution: filename=business_entity_resolution-0.1.0-0.editable-py3-none-any.whl size=1408 sha256=369fa0417fedb7c054668f374811797ca711318159449299dcc6157f43cc642c
  Stored in directory: /tmp/pip-ephem-wheel-cache-4z1sscxr/wheels/a2/83/02/a13f9288c4ae8b84bec6a28a70250587e5ac9da03f88490006
Successfully built business-entity-resolution


In [7]:
!pytest -q

........                                                                 [100%]
8 passed in 0.78s


In [8]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in src:")
print(os.listdir("src"))

print("\nFiles in package:")
print(os.listdir("src/business_entity_resolution"))

Current folder:
/content/amazon-ml-2026-entity-resolution

Files in src:
['business_entity_resolution.egg-info', 'business_entity_resolution']

Files in package:
['evaluation.py', '__init__.py', 'model.py', 'preprocessing.py', 'blocking.py', 'candidate_generation.py', '__pycache__', 'inference.py', 'features.py', 'config.py', 'data_loader.py', 'utils.py']


In [9]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        print(os.path.join(root, file))

./.gitignore
./requirements.txt
./pyproject.toml
./README.md
./.pytest_cache/CACHEDIR.TAG
./.pytest_cache/.gitignore
./.pytest_cache/README.md
./.pytest_cache/v/cache/nodeids
./tests/test_blocking.py
./tests/test_preprocessing.py
./tests/test_features.py
./tests/test_evaluation.py
./tests/__pycache__/test_blocking.cpython-313-pytest-8.4.2.pyc
./tests/__pycache__/test_evaluation.cpython-313-pytest-8.4.2.pyc
./tests/__pycache__/test_preprocessing.cpython-313-pytest-8.4.2.pyc
./tests/__pycache__/test_features.cpython-313-pytest-8.4.2.pyc
./docs/Documentation_template.md
./.git/description
./.git/HEAD
./.git/packed-refs
./.git/index
./.git/config
./.git/refs/remotes/origin/HEAD
./.git/refs/heads/main
./.git/objects/pack/pack-62661abcc6d707b33401a99316b7ea0751da88e4.idx
./.git/objects/pack/pack-62661abcc6d707b33401a99316b7ea0751da88e4.rev
./.git/objects/pack/pack-62661abcc6d707b33401a99316b7ea0751da88e4.pack
./.git/info/exclude
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/pre-rebase.

In [10]:
import os

print("Root files:")
print(os.listdir("."))

Root files:
['.gitignore', '.pytest_cache', 'tests', 'docs', '.git', 'requirements.txt', 'pyproject.toml', 'models', 'notebooks', 'README.md', 'src']


In [11]:
!pip install rapidfuzz scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.4 MB/s eta 0:00:00


In [12]:
%%writefile src/business_entity_resolution/candidate_generation.py

import pandas as pd
def generate_candidates(source1, source2):
    s1 = source1.copy()
    s2 = source2.copy()
    candidate_pairs = set()
    name_index = {}
    for _, row in s2.iterrows():
        key = (
            row["country_block_key"],
            row["name_block_key"]
        )
        if key != ("", ""):
            name_index.setdefault(key, []).append(
                row["entity_id"]
            )
    address_index = {}
    for _, row in s2.iterrows():
        key = (
            row["country_block_key"],
            row["address_block_key"]
        )
        if key != ("", ""):
            address_index.setdefault(key, []).append(
                row["entity_id"]
            )
    for _, row in s1.iterrows():
        name_key = (
            row["country_block_key"],
            row["name_block_key"]
        )
        address_key = (
            row["country_block_key"],
            row["address_block_key"]
        )
        candidates = set()
        candidates.update(
            name_index.get(name_key, [])
        )
        candidates.update(
            address_index.get(address_key, [])
        )
        for source2_id in candidates:
            candidate_pairs.add(
                (
                    row["entity_id"],
                    source2_id
                )
            )
    return pd.DataFrame(
        list(candidate_pairs),
        columns=[
            "source1_id",
            "source2_id"
        ]
    )

Overwriting src/business_entity_resolution/candidate_generation.py
